# INFY Custom Data Source (DB-backed)
This notebook loads INFY daily bars from Postgres via a custom QuantConnect `PythonData` source.

In [1]:
# QuantBook Analysis Tool
qb = QuantBook()

# Force reload so notebook always uses latest db_tick_custom_data.py on disk
import importlib
import db_tick_custom_data
importlib.reload(db_tick_custom_data)
print("db_tick_custom_data loaded from:", db_tick_custom_data.__file__)

DbDailyByTradingsymbol = db_tick_custom_data.DbDailyByTradingsymbol

original_log = DbDailyByTradingsymbol._log.__func__
@classmethod
def quiet_cache_hit_log(cls, message: str) -> None:
    if str(message).startswith("Using existing cache for "):
        return
    return original_log(cls, message)

DbDailyByTradingsymbol._log = quiet_cache_hit_log

db_tick_custom_data loaded from: /Lean/Launcher/bin/Debug/Notebooks/db_tick_custom_data.py


In [18]:
# Request INFY data through custom data subscription
qb.set_start_date(2026, 4, 22)
infy_custom = qb.add_data(DbDailyByTradingsymbol, "INFY", Resolution.DAILY)

# Get 360 daily bars, similar to Cell 2 style
history = qb.history(infy_custom.symbol, 360, Resolution.DAILY)
history.tail()

[DbDailyByTradingsymbol] Refreshing cache for INFY
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=INFY
[DbDailyByTradingsymbol] Resolved instrument_token=408065
[DbDailyByTradingsymbol] Fetched 5747 rows from daily_candles, date range [2003-01-01 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 1154.8/41354744 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/INFY.csv


close    high     low    open  \
symbol                      time                                         
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  1331.0  1309.0  1322.1   
                            2026-04-17  1318.7  1328.3  1306.1  1313.0   
                            2026-04-20  1312.6  1324.9  1308.1  1320.0   
                            2026-04-21  1313.2  1325.0  1299.3  1310.0   
                            2026-04-22  1257.9  1297.7  1257.0  1295.0   

                                         value      volume  
symbol                      time                            
INFY.DbDailyByTradingsymbol 2026-04-16  1319.2  18527029.0  
                            2026-04-17  1318.7  12064048.0  
                            2026-04-20  1312.6   7172300.0  
                            2026-04-21  1313.2  10057440.0  
                            2026-04-22  1257.9  10635696.0

## Universe from sector-tree + index-constituents

This section builds a custom universe by intersecting:
- Sector subtree for `IT Services` (from `sector-tree`)
- Constituents of `BSE 1000` (from `index-constituents`)

The output universe is a list of NSE codes.

In [31]:
# Initialize universe from db_universe_custom_data and simulate 3-year selector behavior

import importlib
from datetime import datetime, timedelta

import pandas as pd

import db_universe_custom_data
import utils

importlib.reload(db_universe_custom_data)
importlib.reload(utils)

# === Load real universe payload from DB ===
print("Loading universe payload from DB...")
session_local = db_universe_custom_data.build_session_local()
payload = db_universe_custom_data.build_universe_payload(session_local)
universe_list = payload.get("universe", [])

if not universe_list:
    raise RuntimeError("DB universe payload is empty.")

print(f"Loaded {len(universe_list)} symbols from DB")




Loading universe payload from DB...
Loaded 58 symbols from DB


In [42]:
# Top-10 by Market Cap for all timestamps
universe_lookup = {row["nse_code"]: row for row in universe_list}

market_cap_by_code = {}
for code, row in universe_lookup.items():
    series = utils.extract_metric_series(
        row.get("processed_financial_data", {}),
        utils.FINANCIAL_SCORE_SECTION,
        utils.FINANCIAL_SCORE_ROW,
    )
    market_cap_by_code[code] = series if series is not None else pd.Series(dtype=float)

market_cap_matrix = pd.DataFrame.from_dict(market_cap_by_code, orient="index")
market_cap_matrix = market_cap_matrix.dropna(axis=1, how="all")

timestamps = pd.to_datetime(market_cap_matrix.columns, errors="coerce")
market_cap_matrix = market_cap_matrix.loc[:, timestamps.notna()]
market_cap_matrix.columns = timestamps[timestamps.notna()]
market_cap_matrix = market_cap_matrix.sort_index(axis=1)

if market_cap_matrix.empty:
    raise RuntimeError("No Market Cap data")

subscribed_symbols = []

for ts in market_cap_matrix.columns:
    ranked = market_cap_matrix[ts].dropna().nlargest(10)

    for symbol in active_subscription_symbols:
        qb.remove_security(symbol)

    active_subscription_symbols = []
    subscribed_symbols = []

    for code, cap in ranked.items():
        security = qb.add_data(DbDailyByTradingsymbol, code, Resolution.DAILY)
        active_subscription_symbols.append(security.symbol)
        subscribed_symbols.append(
            {
                "nse_code": code,
                "symbol": security.symbol,
                "market_cap": float(cap),
            }
        )

    print(str(ts), [(item["nse_code"], item["market_cap"]) for item in subscribed_symbols])

2001-04-01 00:00:00 [('RELIANCE', 42484.2)]
2001-05-01 00:00:00 [('RELIANCE', 48708.0)]
2001-06-01 00:00:00 [('RELIANCE', 45731.4)]
2001-07-01 00:00:00 [('RELIANCE', 39575.25)]
2001-08-01 00:00:00 [('RELIANCE', 38628.15)]
2001-09-01 00:00:00 [('RELIANCE', 32877.9)]
2001-10-01 00:00:00 [('RELIANCE', 31592.55)]
2001-11-01 00:00:00 [('RELIANCE', 35989.8)]
2001-12-01 00:00:00 [('RELIANCE', 38019.3)]
2002-01-01 00:00:00 [('RELIANCE', 37207.5)]
2002-02-01 00:00:00 [('RELIANCE', 38222.25)]
2002-03-01 00:00:00 [('RELIANCE', 37275.15)]
2002-04-01 00:00:00 [('RELIANCE', 34501.5)]
2002-05-01 00:00:00 [('RELIANCE', 32742.6)]
2002-06-01 00:00:00 [('RELIANCE', 33419.1)]
2002-07-01 00:00:00 [('RELIANCE', 30307.2)]
2002-08-01 00:00:00 [('RELIANCE', 31727.85)]
2002-09-01 00:00:00 [('RELIANCE', 32066.1)]
2002-10-01 00:00:00 [('RELIANCE', 32945.55)]
2002-11-01 00:00:00 [('RELIANCE', 35719.2)]
2002-12-01 00:00:00 [('RELIANCE', 37004.55)]
2003-01-01 00:00:00 [('RELIANCE', 34095.6), ('INFY', 27279.14), ('WI

In [ ]:
# Obtain price data for universe securities using history and print it

if not universe_symbols:
    raise RuntimeError("Universe symbols are empty. Run Cell 6 first.")

# Rebind custom data loader to Postgres session used by universe builder.
db_tick_custom_data.SessionLocal = db_universe_custom_data.build_session_local()
DbDailyByTradingsymbol._prepared_symbols = set()

lookback_bars = 60
max_symbols = 20  # keep output manageable
symbols_to_query = universe_symbols[:max_symbols]

print(f"Fetching {lookback_bars} daily bars for {len(symbols_to_query)} securities...")

for symbol in symbols_to_query:
    try:
        raw_hist = qb.history(symbol, lookback_bars, Resolution.DAILY)
        display(raw_hist.head())
    except Exception:
        continue


Fetching 60 daily bars for 10 securities...
[DbDailyByTradingsymbol] Refreshing cache for RELIANCE
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=RELIANCE
[DbDailyByTradingsymbol] Resolved instrument_token=738561
[DbDailyByTradingsymbol] Fetched 6217 rows from daily_candles, date range [2001-04-20 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 1331.0/11593905 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/RELIANCE.csv
Raw history for RELIANCE.DbDailyByTradingsymbol:
                                               close     high      low  \
symbol                          time                                    
RELIANCE.DbDailyByTradingsymbol 2024-02-02  1457.70  1474.90  1433.20   
                                2024-02-05  1439.00  1470.50  1431.50   
                                2024-02-06  1427.80  1441.85  1419.80   
                                2024-02-07  1442.15  1449.50  1429.25   
                         

close     high      low  \
symbol                          time                                    
RELIANCE.DbDailyByTradingsymbol 2024-02-02  1457.70  1474.90  1433.20   
                                2024-02-05  1439.00  1470.50  1431.50   
                                2024-02-06  1427.80  1441.85  1419.80   
                                2024-02-07  1442.15  1449.50  1429.25   
                                2024-02-08  1450.10  1459.50  1427.50   

                                               open    value      volume  
symbol                          time                                      
RELIANCE.DbDailyByTradingsymbol 2024-02-02  1433.20  1457.70  19652588.0  
                                2024-02-05  1460.75  1439.00   8814432.0  
                                2024-02-06  1441.85  1427.80   9047984.0  
                                2024-02-07  1435.90  1442.15   9296568.0  
                                2024-02-08  1450.00  1450.10  14694634.0

[DbDailyByTradingsymbol] Refreshing cache for HDFCBANK
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=HDFCBANK
[DbDailyByTradingsymbol] Resolved instrument_token=341249
[DbDailyByTradingsymbol] Fetched 5747 rows from daily_candles, date range [2003-01-01 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 785.15/26155682 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/HDFCBANK.csv
Raw history for HDFCBANK.DbDailyByTradingsymbol:
                                             close   high    low   open  value  \
symbol                          time                                            
HDFCBANK.DbDailyByTradingsymbol 2024-02-02  723.1  740.4  721.0  737.5  723.1   
                                2024-02-05  722.4  726.0  717.0  723.0  722.4   
                                2024-02-06  722.0  724.8  716.3  722.8  722.0   
                                2024-02-07  715.0  726.9  713.3  726.5  715.0   
                     

close   high    low   open  value  \
symbol                          time                                            
HDFCBANK.DbDailyByTradingsymbol 2024-02-02  723.1  740.4  721.0  737.5  723.1   
                                2024-02-05  722.4  726.0  717.0  723.0  722.4   
                                2024-02-06  722.0  724.8  716.3  722.8  722.0   
                                2024-02-07  715.0  726.9  713.3  726.5  715.0   
                                2024-02-08  701.5  719.5  700.2  714.9  701.5   

                                                volume  
symbol                          time                    
HDFCBANK.DbDailyByTradingsymbol 2024-02-02  44867754.0  
                                2024-02-05  38605046.0  
                                2024-02-06  41075740.0  
                                2024-02-07  54837086.0  
                                2024-02-08  69106124.0

[DbDailyByTradingsymbol] Refreshing cache for BHARTIARTL
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=BHARTIARTL
[DbDailyByTradingsymbol] Resolved instrument_token=2714625
[DbDailyByTradingsymbol] Fetched 5747 rows from daily_candles, date range [2003-01-01 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 1817.0/6220105 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/BHARTIARTL.csv
Raw history for BHARTIARTL.DbDailyByTradingsymbol:
                                                 close     high     low  \
symbol                            time                                   
BHARTIARTL.DbDailyByTradingsymbol 2024-02-02  1150.80  1175.20  1145.3   
                                  2024-02-05  1113.55  1159.65  1110.0   
                                  2024-02-06  1134.05  1156.10  1123.0   
                                  2024-02-07  1134.30  1148.25  1131.8   
                                  2024-02-08  1142.15  

...

close    high      low     open  \
symbol                          time                                            
AXISBANK.DbDailyByTradingsymbol 2024-02-02  1067.05  1096.2  1065.00  1088.05   
                                2024-02-05  1061.50  1079.5  1058.00  1067.05   
                                2024-02-06  1050.05  1065.1  1048.20  1061.55   
                                2024-02-07  1069.10  1073.7  1057.50  1061.05   
                                2024-02-08  1035.90  1075.6  1031.65  1072.00   

                                              value      volume  
symbol                          time                             
AXISBANK.DbDailyByTradingsymbol 2024-02-02  1067.05   8667415.0  
                                2024-02-05  1061.50  12569454.0  
                                2024-02-06  1050.05   8011171.0  
                                2024-02-07  1069.10   7281793.0  
                                2024-02-08  1035.90   9869141.0

[DbDailyByTradingsymbol] Refreshing cache for MARUTI
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=MARUTI
[DbDailyByTradingsymbol] Resolved instrument_token=2815745
[DbDailyByTradingsymbol] Fetched 5618 rows from daily_candles, date range [2003-07-09 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 13060.0/463475 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/MARUTI.csv
Raw history for MARUTI.DbDailyByTradingsymbol:
                                              close      high       low  \
symbol                        time                                       
MARUTI.DbDailyByTradingsymbol 2024-02-02  10655.45  10711.60  10540.40   
                              2024-02-05  10428.55  10759.95  10400.00   
                              2024-02-06  10844.85  10856.75  10458.00   
                              2024-02-07  10935.70  10973.50  10786.80   
                              2024-02-08  10743.90  10951.00  10689.95 

close      high       low  \
symbol                        time                                       
MARUTI.DbDailyByTradingsymbol 2024-02-02  10655.45  10711.60  10540.40   
                              2024-02-05  10428.55  10759.95  10400.00   
                              2024-02-06  10844.85  10856.75  10458.00   
                              2024-02-07  10935.70  10973.50  10786.80   
                              2024-02-08  10743.90  10951.00  10689.95   

                                              open     value    volume  
symbol                        time                                      
MARUTI.DbDailyByTradingsymbol 2024-02-02  10598.00  10655.45  814940.0  
                              2024-02-05  10602.00  10428.55  365105.0  
                              2024-02-06  10499.95  10844.85  829040.0  
                              2024-02-07  10860.00  10935.70  560808.0  
                              2024-02-08  10951.00  10743.90  557082.0

[DbDailyByTradingsymbol] Refreshing cache for TITAN
[DbDailyByTradingsymbol] Resolving instrument for tradingsymbol=TITAN
[DbDailyByTradingsymbol] Resolved instrument_token=897537
[DbDailyByTradingsymbol] Fetched 5747 rows from daily_candles, date range [2003-01-01 -> 2026-04-24]
[DbDailyByTradingsymbol] Latest row close/volume = 4416.0/956942 on 2026-04-24
[DbDailyByTradingsymbol] Wrote cache file: /tmp/lean-db-daily/TITAN.csv
Raw history for TITAN.DbDailyByTradingsymbol:
                                            close     high      low     open  \
symbol                       time                                             
TITAN.DbDailyByTradingsymbol 2024-02-02  3612.40  3713.45  3602.45  3635.00   
                             2024-02-05  3552.05  3624.65  3534.90  3612.95   
                             2024-02-06  3559.15  3577.95  3531.60  3552.05   
                             2024-02-07  3576.50  3620.00  3555.80  3578.00   
                             2024-02-08  3549.1

close     high      low     open  \
symbol                       time                                             
TITAN.DbDailyByTradingsymbol 2024-02-02  3612.40  3713.45  3602.45  3635.00   
                             2024-02-05  3552.05  3624.65  3534.90  3612.95   
                             2024-02-06  3559.15  3577.95  3531.60  3552.05   
                             2024-02-07  3576.50  3620.00  3555.80  3578.00   
                             2024-02-08  3549.10  3610.00  3503.00  3604.80   

                                           value     volume  
symbol                       time                            
TITAN.DbDailyByTradingsymbol 2024-02-02  3612.40  2320153.0  
                             2024-02-05  3552.05  1771614.0  
                             2024-02-06  3559.15  1069901.0  
                             2024-02-07  3576.50   900220.0  
                             2024-02-08  3549.10  1011857.0